# EXEMPLO 2 — RAG-Fusion Básico (sem asyncio)
## Referência Rápida · Aula 7

Versão síncrona simplificada do RAG-Fusion para entender o conceito.
**Para produção**, use a versão com `asyncio` do LAB3.

In [ ]:
# Implementação didática do RRF (sem dependências)

def rrf(rankings: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    """
    Reciprocal Rank Fusion (Cormack et al., 2009)
    rankings: lista de listas de IDs de documentos (ordenados por relevância)
    Retorna: lista de (doc_id, rrf_score) ordenada por score decrescente
    """
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# Exemplo com 3 rankings fictícios
ranking_q1 = ['doc_A', 'doc_B', 'doc_C', 'doc_D']
ranking_q2 = ['doc_B', 'doc_A', 'doc_E', 'doc_C']
ranking_q3 = ['doc_A', 'doc_C', 'doc_B', 'doc_F']

resultado_rrf = rrf([ranking_q1, ranking_q2, ranking_q3])

print('=== RRF: Resultado Final ===')
for rank, (doc_id, score) in enumerate(resultado_rrf, 1):
    print(f'  {rank}º {doc_id}: {score:.5f}')

print()
print('💡 Doc_A aparece em 1º, 2º e 1º → score mais alto!')
print('   Doc_F aparece só em 1 ranking → score mais baixo')

In [ ]:
# Pipeline RAG-Fusion síncrono (para estudo)
# (sem conexão real — substitua pelo retriever real no LAB3)

def rag_fusion_sincrono(query: str, retriever_fn, llm_fn, n: int = 3) -> str:
    """
    Pipeline RAG-Fusion simplificado (síncrono, sem asyncio).
    Bom para entender o conceito; para produção, use a versão async do LAB3.
    """
    # 1. Gerar sub-queries (simplificado — no LAB3 usa o LLM real)
    sub_queries = [f'{query} (perspectiva {i})' for i in range(1, n+1)]
    all_queries = [query] + sub_queries
    
    # 2. Retrieval sequencial (sem paralelismo)
    rankings = [retriever_fn(q) for q in all_queries]
    
    # 3. RRF
    fused = rrf(rankings)
    top_docs = [doc_id for doc_id, _ in fused[:5]]
    
    # 4. Geração
    contexto = f'Documentos: {top_docs}'
    return llm_fn(query, contexto)

# Funções mock para demonstração
def mock_retriever(query: str) -> list[str]:
    import random
    docs = ['doc_001', 'doc_002', 'doc_003', 'doc_004', 'doc_005']
    random.shuffle(docs)
    return docs

def mock_llm(query: str, context: str) -> str:
    return f'[Resposta gerada para: "{query}" | Contexto: {context[:50]}...]'

resultado = rag_fusion_sincrono('Requisitos prisão preventiva', mock_retriever, mock_llm)
print(f'\n=== Resultado RAG-Fusion (mock) ===')
print(resultado)
print()
print('→ Para implementação real com vLLM + OpenSearch, ver LAB3_RAG_Fusion_Completo.ipynb')